# Feature Engineering

### Objective

The objective of this notebook is to transform the cleaned dataset into machine learning-ready features.

In this notebook, we will:

- Select the required features
- Perform NLP preprocessing
- Convert text into numerical features using TF-IDF Vectorization
- Encode the target variable
- Split the dataset into training and testing sets
- Save preprocessing artifacts for model training

Dataset Used:

- cleaned_data.csv

In [64]:
import warnings
warnings.filterwarnings("ignore")

In [65]:
import pandas as pd
import numpy as np

In [66]:
import re
import string
import joblib

In [67]:
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [69]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vmangla/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/vmangla/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/vmangla/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [70]:
df = pd.read_csv("../dataset/cleaned_data.csv")
df.head()

,Call_Timestamp,Patient_Age,Patient_Gender,State,Language,New_or_Existing_Patient,Call_Channel,Caller_Relationship,Chief_Complaint,Symptoms_Reported,...,Chronic_Condition,Insurance_Type,Preferred_Appointment_Urgency,Prior_Hospital_Visit_30D,Call_Duration_Minutes,Wait_Time_Minutes,Repeat_Caller,Sentiment,Call_Transcript_Summary,Recommended_Department
0,2025-02-27 20:09:18,33,Female,Alaska,English,Existing,Phone,Self,Needs to see a doctor and agent every,mild fever; blood pressure follow-up; general ...,...,Hypertension,Medicaid,Immediate,Yes,17,1,No,Anxious,Caller reached the hospital call center genera...,Primary Care
1,2025-06-27 20:33:38,27,Male,Louisiana,English,Existing,Phone,Self,"Caller reports abdominal pain, vomiting",abdominal pain; vomiting; acid reflux,...,No Chronic Condition,Private,Routine,No,9,5,No,Calm,Family member called the access center stomach...,Gastroenterology
2,2026-01-09 06:34:10,38,Male,Arizona,English,New,Phone,Caregiver,"Caller reports needs specialist slot, book app...",needs specialist slot; book appointment,...,No Chronic Condition,Private,Routine,No,11,3,No,Anxious,Caller reached the hospital call center appoin...,Scheduling
3,2025-09-01 10:33:37,7,Female,Connecticut,English,Existing,Phone,Parent,"Patient is dealing with skin lesion, acne flare",skin lesion; acne flare,...,Multiple,Medicaid,Soon,No,25,9,No,Distressed,Family member called the access center dermato...,Dermatology
4,2024-10-07 16:00:57,17,Male,Iowa,English,Existing,Phone,Self,"Parent says child has fatigue, weight loss",fatigue; weight loss,...,Multiple,Private,Soon,Yes,12,9,No,Distressed,Family member called the access center oncolog...,Pediatrics


In [71]:
df.shape

(10000, 24)

In [72]:
X = df["Chief_Complaint"]
y = df["Recommended_Department"]

X.shape, y.shape

((10000,), (10000,))

# NLP Preprocessing

In [73]:
# Create Stopword List
stop_words = set(stopwords.words("english"))

In [74]:
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

In [75]:
# Create Preprocessing Function

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove extra spaces
    text = " ".join(text.split())

    # Tokenization
    words = text.split()

    # Remove stopwords and apply lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [76]:
X_clean = X.apply(preprocess_text)

In [77]:
comparison = pd.DataFrame({
    "Original": X.head(10),
    "Processed": X_clean.head(10)
})

comparison

,Original,Processed
0,Needs to see a doctor and agent every,need see doctor agent every
1,"Caller reports abdominal pain, vomiting",caller report abdominal pain vomiting
2,"Caller reports needs specialist slot, book app...",caller report need specialist slot book appoin...
3,"Patient is dealing with skin lesion, acne flare",patient dealing skin lesion acne flare
4,"Parent says child has fatigue, weight loss",parent say child fatigue weight loss
5,"Caller reports labor symptoms, vaginal bleeding",caller report labor symptom vaginal bleeding
6,General health concern and writer policy,general health concern writer policy
7,Parent calling about payment plan request,parent calling payment plan request
8,Concern about insurance problem,concern insurance problem
9,Patient is dealing with mild shortness of brea...,patient dealing mild shortness breath fatigue


In [78]:
X_clean.head()

0                          need see doctor agent every
1                caller report abdominal pain vomiting
2    caller report need specialist slot book appoin...
3               patient dealing skin lesion acne flare
4                 parent say child fatigue weight loss
Name: Chief_Complaint, dtype: object

# TF-IDF Vectorization

In [79]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(X_clean)
X_tfidf.shape

(10000, 5000)

In [80]:
feature_names = tfidf.get_feature_names_out()
feature_names[:30]

array(['abdominal', 'abdominal pain', 'ability', 'ability car',
       'ability cultural', 'ability front', 'ability poor',
       'ability sign', 'ability skin', 'ability take', 'ability type',
       'able', 'able cover', 'able live', 'able main', 'able official',
       'able power', 'accept', 'accept adult', 'accept check',
       'accept choose', 'accept create', 'accept east', 'accept goal',
       'accept mean', 'accept resource', 'accept stop', 'accept tonight',
       'accept water', 'according'], dtype=object)

In [81]:
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=feature_names)
tfidf_df.head()

,abdominal,abdominal pain,ability,ability car,ability cultural,ability front,ability poor,ability sign,ability skin,ability take,...,yet building,yet far,yet fly,yet friend,yet goal,yet may,yet recent,young,young drive,young get
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.370753,0.370753,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Label Encoding

In [82]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded[:10]

array([13,  4, 15,  2, 11,  8, 13,  0,  0,  7])

In [83]:
label_encoder.classes_

array(['Billing', 'Cardiology', 'Dermatology', 'Emergency Department',
       'Gastroenterology', 'Mental Health', 'Neurology', 'Nurse Triage',
       'OB-GYN', 'Oncology', 'Orthopedics', 'Pediatrics', 'Pharmacy',
       'Primary Care', 'Pulmonology', 'Scheduling', 'Urgent Care'],
      dtype=object)

In [84]:
mapping = pd.DataFrame({
    "Department": label_encoder.classes_,
    "Encoded Label": range(len(label_encoder.classes_))
})
mapping

,Department,Encoded Label
0,Billing,0
1,Cardiology,1
2,Dermatology,2
3,Emergency Department,3
4,Gastroenterology,4
5,Mental Health,5
6,Neurology,6
7,Nurse Triage,7
8,OB-GYN,8
9,Oncology,9


# Train-Test Split

In [85]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 5000), (2000, 5000), (8000,), (2000,))

# Save Artifacts

In [86]:
print(joblib.dump(label_encoder, "../artifacts/label_encoder.pkl"))
print(joblib.dump(tfidf, "../artifacts/tfidf_vectorizer.pkl"))

['../artifacts/label_encoder.pkl']
['../artifacts/tfidf_vectorizer.pkl']


In [87]:
print(joblib.dump(X_train, "../artifacts/X_train.pkl"))
print(joblib.dump(X_test, "../artifacts/X_test.pkl"))
print(joblib.dump(y_train, "../artifacts/y_train.pkl"))
print(joblib.dump(y_test, "../artifacts/y_test.pkl"))

['../artifacts/X_train.pkl']
['../artifacts/X_test.pkl']
['../artifacts/y_train.pkl']
['../artifacts/y_test.pkl']
